In [1]:
import sys
import os
import glob 
import time
os.environ['CUDA_VISIBLE_DEVICES']=""
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from collections import defaultdict, Counter
import numpy as np
import dask.dataframe as dd
from matplotlib import pyplot as plt
from tqdm import tqdm
from joblib import Parallel, delayed
from scipy.stats import boxcox
from scipy import stats
import polars as pl
import pandas as pd
import re
from itertools import groupby
from datetime import datetime, timedelta
import pyarrow.dataset as ds
from pathlib import Path

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
EMB_PATH = Path('../../../data/embeddings/')

# AGE

In [3]:
def log_log_transform(y, epsilon=1e-10):
    """Apply log(log(y)) transformation for doubly exponential data"""
    # Ensure y > 0 for double log
    y_positive = y - y.min() + 1  # Shift to ensure y > 0
    return np.log(np.log(y_positive + epsilon))

In [4]:
def age_anomaly_target(df):
    df = df.copy()

    # считаем mean, std для cv 
    df_mean = df['post_amount'].apply(lambda x: np.mean(x) if len(x) > 0 else np.nan)
    df_std = df['post_amount'].apply(lambda x: np.std(x) if len(x) > 0 else np.nan)

    # cчитаем CV
    df["cv"] = df_std / df_mean

    # yаходим пороги
    q95_cv = df["cv"].quantile(0.95)

    # если аномалия чисто по cv плохая можно попробовать дополнительно к cv взять q95 или q05
    #q05_mean = df_mean.quantile(0.05) # Добавим условие низкого чека, как мы обсуждали

    # cоздаем таргет 
    df["target_anomaly"] = (
        (df["cv"] > q95_cv)
    ).astype(int)

    return df["target_anomaly"]


In [5]:
def get_regression_target(interval_feature: pd.Series, target_feature: pd.Series, horizon: int = 30):
    return [
        np.log1p(
            np.sum(np.asarray(a)[(d - d[0]) < horizon])
        ) for a, d in zip(
            target_feature, 
            interval_feature
        )
    ]

In [6]:
embeddings_path = EMB_PATH / 'coles_embs' / 'embeddings_age_coles_ntp_TILL.parquet'
embeddings_path_prepocessed = '../../../data/embeddings/preprocessed_coles_embs/embeddings_age_coles.parquet'
embeddings_df = pd.read_parquet(embeddings_path).dropna()

embeddings_df['reg_target'] = get_regression_target(
    interval_feature=embeddings_df.post_trans_date,
    target_feature=embeddings_df.post_amount_rur,
    horizon=30 # month for AGE dataset
)

# Leave it as it was
embeddings_df['post_forecast_target'] = ([ np.sum(v == v[0])  for v in embeddings_df['post_trans_date'].values])
y = embeddings_df['post_forecast_target']
y_transformed = np.log1p(y)
embeddings_df['post_forecast_target'] =  y_transformed
embeddings_df['post_amount'] = np.array(embeddings_df['post_amount_rur'].values)
embeddings_df['post_target'] = np.array(embeddings_df['target'].values)
#post_group is ntp target
embeddings_df['post_group'] = np.array(embeddings_df['post_small_group'].values)
embeddings_df.drop(['post_small_group'], axis=1, inplace=True)
embeddings_df['post_anomaly_target'] = age_anomaly_target(embeddings_df)
embeddings_df.to_parquet(embeddings_path_prepocessed)

KeyboardInterrupt: 

In [ ]:
embeddings_path = EMB_PATH / 'coles_embs' / 'embeddings_age_coles_ntp_TILL.parquet'
embeddings_path_prepocessed = EMB_PATH / 'preprocessed_coles_embs' / 'embeddings_age_coles.parquet'
#embeddings_df = parquet_df[parquet_df['used_in_train'] == 0]
embeddings_df = pd.read_parquet(embeddings_path).dropna()
post_trans_date = embeddings_df['post_trans_date'].values

embeddings_df['post_forecast_target'] = ([ np.sum(v == v[0])  for v in embeddings_df['post_trans_date'].values])
y = embeddings_df['post_forecast_target']
y_transformed = np.log1p(y)
embeddings_df['post_forecast_target'] =  y_transformed
#embeddings_df['post_forecast_target'] = boxcox(embeddings_df['post_forecast_target'].values)[0]
embeddings_df['post_amount'] = np.array(embeddings_df['post_amount_rur'].values)
embeddings_df.drop(['post_amount_rur'], axis=1, inplace=True)
embeddings_df['post_target'] = np.array(embeddings_df['target'].values)
#post_group is ntp target
embeddings_df['post_group'] = np.array(embeddings_df['post_small_group'].values)
embeddings_df.drop(['post_small_group'], axis=1, inplace=True)
embeddings_df['post_anomaly_target'] = age_anomaly_target(embeddings_df)
embeddings_df.to_parquet(embeddings_path_prepocessed)

In [ ]:
#plt.hist(embeddings_df['embedding'].iloc[1])

In [ ]:
transformations = {
    'original': embeddings_df['post_forecast_target'],
    'log1p': np.log1p(embeddings_df['post_forecast_target']),
    'sqrt': np.sqrt(embeddings_df['post_forecast_target']),
    'cube_root': embeddings_df['post_forecast_target'] ** (1/3),
    'fourth_root': embeddings_df['post_forecast_target'] ** 0.25,
    'log_log': np.log(np.log1p(embeddings_df['post_forecast_target'] + 1e-10)),
}

# Plot each transformation
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for (name, data), ax in zip(transformations.items(), axes.flatten()):
    ax.hist(data, bins=50, alpha=0.7)
    ax.set_title(f'{name} (skew: {stats.skew(data):.2f})')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
plt.hist(boxcox(embeddings_df['post_forecast_target']),bins='auto')

In [ ]:
plt.hist([ np.sum(v == v[0])  for v in embeddings_df['post_trans_date'].values])

In [ ]:
plt.hist(AGE_DF['post_forecast_target'])

# taobao

In [4]:
def get_taobao_anomaly(path):
    # предсказание поведения пользователя: кладет товар в корзину, но не покупает его
    embed_taobao = pl.scan_parquet(path)
    coll_embed_taobao = embed_taobao.collect()
    df_seq = coll_embed_taobao.to_pandas().copy().dropna()
    # какие колонки хотим развернуть в обычный вид
    cols_to_explode = ["time", "behavior_type", "item_id"]

    # разворачиваем построчно
    pd_df_concat = (
        df_seq[["user_id"] + cols_to_explode]
        .explode(cols_to_explode)
        .reset_index(drop=True)
    )

    # приводим время и сортируем
    pd_df_concat["time"] = pd.to_datetime(pd_df_concat["time"])
    pd_df_concat = pd_df_concat.sort_values(["user_id", "time"]).reset_index(drop=True)

    ITEM_COL = "item_id"

    # 4 — добавление в корзину
    df_add_item = (
        pd_df_concat[pd_df_concat["behavior_type"] == 4]
        [["user_id", ITEM_COL, "time"]]
        .rename(columns={"time": "timestamp_add"})
    )

    # 3 — покупка
    df_buy_item = (
        pd_df_concat[pd_df_concat["behavior_type"] == 3]
        [["user_id", ITEM_COL, "time"]]
        .rename(columns={"time": "timestamp_buy"})
    )

    # для каждой пары (user, item) смотрим, был ли add без buy
    merged_item = df_add_item.merge(
        df_buy_item,
        on=["user_id", ITEM_COL],
        how="left"
    )

    # аномалия: добавил в корзину, но не купил
    anom_item = merged_item[merged_item["timestamp_buy"].isna()]

    print("Аномальных (client, item):", len(anom_item))

    # пары (user, item), где есть такая аномалия
    anom_pairs = anom_item[["user_id", ITEM_COL]].drop_duplicates()
    anom_pairs["anomaly_item_cart_no_buy"] = True

    # добавляем флаг
    pd_df_concat = pd_df_concat.merge(
        anom_pairs,
        on=["user_id", ITEM_COL],
        how="left"
    )

    pd_df_concat["anomaly_item_cart_no_buy"] = (
        pd_df_concat["anomaly_item_cart_no_buy"]
        .fillna(False)
    )

    # момент первой аномальной корзины по пользователю
    first_cart_time_item = (
        anom_item
        .groupby("user_id")["timestamp_add"]
        .min()
        .rename("first_anomaly_time_item")
    )

    pd_df_concat = pd_df_concat.merge(first_cart_time_item, on="user_id", how="left")

    # фаза относительно первой аномалии
    pd_df_concat["trget_anomaly_cart_no_buy"] = np.where(
        pd_df_concat["first_anomaly_time_item"].notna() &
        (pd_df_concat["time"] >= pd_df_concat["first_anomaly_time_item"]),
        "post",
        "pre"
    )

    pd_df_concat = pd_df_concat.sort_values(["user_id", "time"]).reset_index(drop=True)

    # возвращаем в изначальный вид
    phase_seq = (
        pd_df_concat
        .groupby("user_id")["trget_anomaly_cart_no_buy"]
        .apply(list)                      
        .reset_index()
    )

    phase_seq = phase_seq.rename(
        columns={"trget_anomaly_cart_no_buy": "trget_anomaly_cart_no_buy_seq"}
    )
    df_seq = df_seq.merge(phase_seq, on="user_id", how="left")
    post_anomaly_target = [ int('post' in v) for v in df_seq['trget_anomaly_cart_no_buy_seq'].values]
    return post_anomaly_target

In [5]:
def check_anomaly(row):

    behavior_type = row["post_behavior_type"]
    item_id = row["post_item_id"]

    add_cart_count = {}
    purchase = set()

    for type, item in zip(behavior_type, item_id):
        if type == 3:
            add_cart_count[item] = add_cart_count.get(item, 0) + 1

        elif type == 4:
            purchase.add(item)

    is_anoamly = 0

    for item, count in add_cart_count.items():
        if count > 1 and item not in purchase:
            is_anoamly =1
            break

    return is_anoamly

In [6]:
def get_taobao_forecast(path):
    embed_taobao = pl.scan_parquet(path)
    coll_embed_taobao = embed_taobao.collect()
    df_fore = coll_embed_taobao.to_pandas().copy().dropna()
    cols_to_explode = ['time', "item_id", "behavior_type"]
    df_fore_exp = (
        df_fore[["user_id"] + cols_to_explode]
        .explode(cols_to_explode)
        .reset_index(drop=True)
    )
    df_fore_exp["time"] = pd.to_datetime(df_fore_exp["time"])
    df_fore_exp = df_fore_exp.sort_values(["user_id", "time"]).reset_index(drop=True)
    is_transaction = df_fore_exp["behavior_type"] == 3
    df_fore_exp["last_trx_time"] = np.where(is_transaction, df_fore_exp["time"], pd.NaT)
    df_fore_exp["last_trx_time"] = (
        df_fore_exp
        .groupby("user_id")["last_trx_time"]
        .ffill()
    )
    df_fore_exp["last_trx_time"] = pd.to_datetime(df_fore_exp["last_trx_time"])
    delta = df_fore_exp["time"] - df_fore_exp["last_trx_time"]
    df_fore_exp["time_since_last_tx_days"] = delta.dt.total_seconds() / (3600 * 24)
    target_forecast_df = df_fore_exp[
        df_fore_exp["time_since_last_tx_days"].notna()
        & (df_fore_exp["time_since_last_tx_days"] > 0)
    ]
    target_seq = (
        df_fore_exp
        .groupby("user_id")["time_since_last_tx_days"]
        .apply(list)                      
        .reset_index()
    )

    df_fore_final = (
        df_fore
        .merge(target_seq, on="user_id", how="left")
    )
    return np.stack(df_fore_final["time_since_last_tx_days"])

In [22]:
users = pd.read_csv(EMB_PATH / '../taobao/tianchi_mobile_recommend_train_user.csv')
item_to_category = {
    str(k): str(v) for k, v in zip(users['item_id'], users['item_category'])
}

In [23]:
def remove_consecutive_duplicates(lst):
    """Remove consecutive duplicates from a list"""
    if not lst:
        return []
    result = [lst[0]]
    for i in range(1, len(lst)):
        if lst[i] != lst[i-1]:
            result.append(lst[i])
    return result

In [24]:
def map_without_consecutive_duplicates(item_list):
    categories = []
    last_category = None
    for item_id in item_list:
        category = item_to_category.get(item_id)
        if category is not None and category != last_category:
            categories.append(category)
            last_category = category
    return categories

In [25]:
def taobao_reg_target(data_time, unit="h", horizon=300):
    """
    log(1 + number of events in the first `horizon` time units after first post event)
    unit: "h" (hours), "D" (days), "m" (minutes), "s" (seconds)
    """
    return np.array([
        np.log1p(np.sum(((t - t[0]) / np.timedelta64(1, unit)) < horizon)) if len(t) else 0.0
        for t in data_time
    ])


In [26]:
embeddings_path = EMB_PATH / 'coles_embs' / 'embeddings_taobao_coles_ntp_TILL_best.parquet'

parquet_df = pd.read_parquet(embeddings_path).dropna()#.head(1000)
embeddings_df = parquet_df.copy()
embeddings_df['post_target'] = list(embeddings_df['target'].values)
# embeddings_df['post_amount'] = embeddings_df['post_behavior_type'] #[np.zeros_like(t).astype('int') for t in embeddings_df['post_time'].values]
embeddings_df["reg_target"] = taobao_reg_target(embeddings_df["post_time"], unit="h", horizon=300)
#embeddings_df['post_anomaly_target']  = get_taobao_anomaly(embeddings_path)
embeddings_df['post_anomaly_target'] = [check_anomaly(r[1]) for r in list(embeddings_df.head(None).iterrows())]
embeddings_df['post_trans_date'] = np.array(embeddings_df['post_time'].values)
embeddings_df['client_id'] = np.array(embeddings_df['user_id'].values)

post_forecast_target = get_taobao_forecast(embeddings_path)
embeddings_df['post_forecast_target'] = [np.nanmedian(v) for v in post_forecast_target]

# Map to categories without duplicates
embeddings_df['post_group'] = embeddings_df['post_item_id'].apply(map_without_consecutive_duplicates)

# Use raw item IDs instead of categories
#embeddings_df['post_group'] = embeddings_df['post_item_id'].apply(lambda lst: [key for key, _ in groupby(lst)])

#embeddings_df = embeddings_df[embeddings_df['post_group'].apply(len) > 0].copy()
#embeddings_df = embeddings_df.dropna()
embeddings_path_prepocessed = EMB_PATH / "preprocessed_coles_embs" / "embeddings_taobao_coles.parquet"
embeddings_df.to_parquet(embeddings_path_prepocessed)

/usr/local/lib/python3.10/dist-packages/numpy/lib/nanfunctions.py:1217: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a, func=_nanmedian, keepdims=keepdims,


# Rossman

In [12]:
def rossman_reg_target(sales: pd.Series, horizon=30):
    future_sum = sales.apply(lambda x: np.sum(x[:horizon]))

    mu = future_sum.mean()
    sigma = future_sum.std()

    return (future_sum - mu) / sigma

In [13]:
embeddings_path = EMB_PATH / 'coles_embs' / 'embeddings_rossman_coles_ntp_TILL_best_lied.parquet'
max_users = None
parquet_df = pd.read_parquet(embeddings_path).dropna().head(max_users)
embeddings_df = parquet_df.copy()
embeddings_df['post_trans_date'] = np.array(embeddings_df['post_Date'].values)
embeddings_df['client_id'] = np.array(embeddings_df['Store'].values)
embeddings_df['post_amount'] = [np.zeros_like(t).astype('int') for t in embeddings_df['post_trans_date'].values]
store_info_df = pd.read_csv(EMB_PATH / '../rossman' / 'store.csv')
store_type_map = dict(zip(store_info_df['Store'], store_info_df['StoreType']))
embeddings_df['store_type_letter'] = embeddings_df['client_id'].map(store_type_map)
# Convert to integer codes
embeddings_df['post_target'] = pd.factorize(embeddings_df['store_type_letter'])[0]
embeddings_df['post_forecast_target'] = np.log1p([np.median(v) for v in embeddings_df['post_Sales']])

embeddings_df['reg_target'] = rossman_reg_target(embeddings_df['Sales'], horizon=30)
embeddings_path_prepocessed = EMB_PATH / "preprocessed_coles_embs" / "embeddings_rossman_coles.parquet"
embeddings_df.to_parquet(embeddings_path_prepocessed)

# Favorita

In [145]:

def post_sales_reg_target(
    post_date: pd.Series,
    df: pd.DataFrame,
    horizon: int = 30,
):
    sales_cols = [c for c in df.columns if c.startswith("post_class_") and c.endswith("_sales")]

    return np.array([
        np.log1p(
            sum(
                np.sum(np.asarray(row[c])[((d - d[0]) / np.timedelta64(1, "D")) <= horizon])
                for c in sales_cols
            )
        ) if len(d) else 0.0
        for d, row in zip(post_date, df[sales_cols].to_dict("records"))
    ])

In [ ]:
embeddings_path = EMB_PATH / 'coles_embs' / 'embeddings_favorita_coles.parquet'
parquet_df = pd.read_parquet(embeddings_path)
embeddings_df = parquet_df.copy()
embeddings_df['post_trans_date'] = embeddings_df['post_date']
embeddings_df['client_id'] = embeddings_df['store_nbr']
embeddings_df['post_amount'] = post_sales_reg_target(embeddings_df.post_date, embeddings_df, horizon=30)

# Convert to integer codes
embeddings_df['post_target'] = (embeddings_df['post_amount'] < 12).to_numpy(dtype=int)
# embeddings_df['post_forecast_target'] = np.log1p([np.median(v) for v in embeddings_df['post_Sales']])

embeddings_path_prepocessed = EMB_PATH / 'preprocessed_coles_embs' / "embeddings_favorita_coles.parquet"
embeddings_df.to_parquet(embeddings_path_prepocessed)

# Twitter (in progress)

In [8]:
embeddings_path = EMB_PATH / 'coles_embs' / 'embeddings_twitter'
embeddings_df = pd.read_parquet(embeddings_path).dropna()
embeddings_df['post_trans_date'] = embeddings_df['post_char_number']
embeddings_df['client_id'] = embeddings_df['tweet_id']
# Convert to integer codes
embeddings_df['post_target'] = embeddings_df['sentiment']
embeddings_df['post_group'] = embeddings_df['post_char_number']
embeddings_df['post_amount'] = embeddings_df['likes']
embeddings_path_prepocessed = EMB_PATH / 'preprocessed_coles_embs' / "embeddings_twitter.parquet"
embeddings_df.to_parquet(embeddings_path_prepocessed)

In [7]:
embeddings_df[embeddings_df['tweet_id'] == 832].retweets

1915    86
Name: retweets, dtype: int64